# `extract_cargadores.ipynb` - Extraccion resiliente de puntos de recarga (Open Charge Map)

Requisito: *"Ingestion dinamica mediante peticiones HTTP REST a la API de Open Charge Map... Se deben extraer latitud, longitud y potencia (kW) en un radio de 20 km del centro de Sevilla."*

`obtener_estaciones_carga()` intenta la llamada HTTP hasta `API_MAX_REINTENTOS + 1` veces. Igual que en `demografia.ipynb` del proyecto GeoStat, se contemplan dos formas de fallo:

1. **Fallo de conexion / HTTP**: timeout, DNS, codigo de error -> `requests.exceptions.RequestException`.
2. **Fallo "silencioso"**: la API responde `200 OK` pero con un cuerpo que no es la lista de puntos esperada.

Si tras los reintentos no hay respuesta valida, se activa `ESTACIONES_FALLBACK` (de `config.ipynb`) y el pipeline continua. Sin API, Sevilla tiene del orden de 20-30 estaciones conocidas; con la API en vivo (clave propia, `maxresults=1000`) se obtienen los puntos reales dentro del radio de 20 km - en la ultima ejecucion, 533 puntos, de los cuales 352 caen dentro de los 11 distritos oficiales y 181 quedan en cuarentena espacial (municipios colindantes, Isla de la Cartuja). El numero exacto crece con el tiempo, no es una cifra fija.

Dos particularidades de la propia API, ya resueltas en el codigo:

- **User-Agent propio** (`USER_AGENT_API`, definido en `config.ipynb`): Open Charge Map devuelve `403 Forbidden` a los User-Agent genericos de libreria (`python-requests/x.x`), por tratarlos como trafico de robot.
- **Clave obligatoria a partir de `maxresults > 250`**: se envia como parametro `key` en la URL (mas fiable frente a proxys que recortan cabeceras personalizadas); su valor por defecto ya viene integrado en `config.ipynb` (`API_KEY_OPENCHARGEMAP`), para no depender de una variable de entorno que un kernel de Jupyter ya abierto no siempre hereda.

> Depende de `config.ipynb` y `logging_config.ipynb` (usa `logger`).

In [1]:
import os
if os.path.basename(os.getcwd()) == "geokw_etl":
    os.chdir("..")

%run geokw_etl/config.ipynb
%run geokw_etl/logging_config.ipynb

Tabla maestra cargada: 11 distritos. poblacion total 697.233 hab.
Dataset de respaldo cargado: 31 estaciones
2026-09-17 09:13:18 - INFO - Prueba de logging desde logging_config.ipynb


In [2]:
import logging

import requests

## Función principal

In [3]:
def obtener_estaciones_carga():
    """
    Intenta obtener la lista de puntos de recarga desde la API de Open
    Charge Map en un radio de RADIO_KM alrededor del centro de Sevilla.
    Si falla la conexion o la respuesta no tiene el formato esperado,
    cae automaticamente al dataset de respaldo interno (ESTACIONES_FALLBACK).

    Devuelve una tupla (lista_de_estaciones, origen) donde cada estacion
    es un dict {"nombre", "lat", "lon", "potencia_kw"} y origen es
    'API' o 'FALLBACK'.
    """
    intentos = 0
    while intentos <= API_MAX_REINTENTOS:
        intentos += 1
        try:
            logger.info(
                f"Intento {intentos}/{API_MAX_REINTENTOS + 1} de conexion a la API "
                f"de Open Charge Map: {API_OPENCHARGEMAP_URL}"
            )
            params = {
                "output": "json",
                "countrycode": "ES",
                "latitude": SEVILLA_CENTRO_LAT,
                "longitude": SEVILLA_CENTRO_LON,
                "distance": RADIO_KM,
                "distanceunit": "KM",
                "maxresults": API_MAX_RESULTADOS,
                "compact": "true",
                "verbose": "false",
            }
            if API_KEY_OPENCHARGEMAP:
                params["key"] = API_KEY_OPENCHARGEMAP

            headers = {"User-Agent": USER_AGENT_API, "Accept": "application/json"}

            respuesta = requests.get(
                API_OPENCHARGEMAP_URL, params=params, headers=headers,
                timeout=API_TIMEOUT_SEGUNDOS,
            )
            respuesta.raise_for_status()
            cuerpo = respuesta.json()

            if not isinstance(cuerpo, list) or len(cuerpo) == 0:
                raise ValueError(
                    "La API respondio pero el cuerpo no tiene el formato "
                    "esperado (lista de puntos de recarga)."
                )

            estaciones = []
            for poi in cuerpo:
                direccion = poi.get("AddressInfo") or {}
                lat = direccion.get("Latitude")
                lon = direccion.get("Longitude")
                if lat is None or lon is None:
                    continue

                conexiones = poi.get("Connections") or []
                potencias = [con.get("PowerKW") for con in conexiones if con.get("PowerKW")]
                # Potencia de la estacion = la del conector mas rapido disponible.
                # Si ningun conector reporta potencia, se asume un valor
                # conservador (carga lenta tipo Schuko).
                potencia_kw = max(potencias) if potencias else 3.7

                estaciones.append({
                    "nombre": direccion.get("Title", "Punto de recarga sin nombre"),
                    "lat": lat,
                    "lon": lon,
                    "potencia_kw": potencia_kw,
                })

            if not estaciones:
                raise ValueError("La API respondio correctamente pero sin puntos de recarga utilizables.")

            logger.info(f"Puntos de recarga obtenidos desde la API: {len(estaciones)}")
            return estaciones, "API"

        except (requests.exceptions.RequestException, ValueError) as error:
            logger.warning(f"Fallo al obtener puntos de recarga de la API: {error}")

    logger.warning(
        "No fue posible obtener puntos de recarga de la API tras "
        f"{API_MAX_REINTENTOS + 1} intentos. Se activa el dataset de "
        "respaldo interno (fallback) para no detener la ejecucion."
    )
    return list(ESTACIONES_FALLBACK), "FALLBACK"

## Prueba rápida

Llamada real, para ver en vivo los `WARNING` (si la API falla) y el origen final de los datos.

In [4]:
estaciones_prueba, origen_prueba = obtener_estaciones_carga()

print(f"\nOrigen: {origen_prueba}")
print(f"Estaciones disponibles: {len(estaciones_prueba)}")
print("Ejemplo:", estaciones_prueba[:3])


Origen: API
Estaciones disponibles: 533
Ejemplo: [{'nombre': 'Parking Albareda', 'lat': 37.38958069996167, 'lon': -5.995750113896634, 'potencia_kw': 7}, {'nombre': 'Parking Plaza Nueva', 'lat': 37.38906332169779, 'lon': -5.995629900618383, 'potencia_kw': 22}, {'nombre': 'Parking APK2 Magdalena', 'lat': 37.3907947, 'lon': -5.997415199999978, 'potencia_kw': 7.4}]


## Diagnóstico: fecha de alta de las estaciones (`DateCreated`)

Open Charge Map es una base de datos colaborativa en vivo: altas, bajas y verificaciones de puntos ocurren continuamente, así que dos ejecuciones en fechas distintas pueden devolver totales distintos de forma legítima — no es un fallo del pipeline. Cada punto trae un campo `DateCreated` con la fecha exacta de alta en la base de datos, que permite auditar cuándo se incorporó cada estación.

Es un script de diagnóstico puntual, no forma parte del pipeline en sí — `obtener_estaciones_carga()` no guarda `DateCreated` en el resultado final, solo `nombre`/`lat`/`lon`/`potencia_kw`.

> **Criterio de interpretación**: este campo sirve para explicar diferencias *pequeñas* (unas pocas estaciones) entre ejecuciones cercanas en el tiempo. Una diferencia de decenas o cientos de estaciones entre dos ejecuciones (p. ej. 350 frente a 533) casi nunca se debe a altas reales en la base de datos — ese ritmo no es realista para una sola ciudad en pocos días. Ante una discrepancia grande, lo primero a revisar es la configuración (si la clave llegó realmente a la petición, el valor de `API_MAX_RESULTADOS`), no la fecha de alta de las estaciones.

In [5]:
import pandas as pd

respuesta = requests.get(API_OPENCHARGEMAP_URL, params={
    "output": "json", "countrycode": "ES",
    "latitude": SEVILLA_CENTRO_LAT, "longitude": SEVILLA_CENTRO_LON,
    "distance": RADIO_KM, "distanceunit": "KM",
    "maxresults": API_MAX_RESULTADOS, "compact": "true", "verbose": "false",
    "key": API_KEY_OPENCHARGEMAP,
}, headers={"User-Agent": USER_AGENT_API, "Accept": "application/json"})

cuerpo = respuesta.json()
fechas = pd.DataFrame([
    {
        "nombre": (poi.get("AddressInfo") or {}).get("Title"),
        "fecha_creacion": poi.get("DateCreated"),
    }
    for poi in cuerpo
])
fechas["fecha_creacion"] = pd.to_datetime(fechas["fecha_creacion"])

recientes = fechas[fechas["fecha_creacion"] >= pd.Timestamp.now(tz="UTC") - pd.Timedelta(days=4)]
print(f"Estaciones dadas de alta en los ultimos 4 dias: {len(recientes)}")
print(recientes.sort_values("fecha_creacion", ascending=False).to_string(index=False))

Estaciones dadas de alta en los ultimos 4 dias: 2
                                   nombre            fecha_creacion
                        McDonalds Sevilla 2026-09-15 08:06:00+00:00
Ayunt. Sevilla: Calle Baltasar de Alcázar 2026-09-15 08:04:00+00:00


---
✅ **Extracción de puntos de recarga verificada.**